# Make a virtual forecast model run collection (FMRC)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-icechunks/2026-ioos-hackathon/blob/main/notebooks/gomofs-fmrc-icechunk.ipynb)

NOAA's Gulf of Maine ROMS model (GoMOFS) runs four times a day. Each run, or *cycle*, writes
NetCDF files with one time step each: two nowcast files and a 72-hour forecast every 3 hours
(24 files). We use the nowcast file valid at the cycle time, `n006`, plus the 24 forecast files:
25 files for each cycle. We take three cycles, one day apart, and keep **all** of them in one Icechunk store
on your own disk, with two time dimensions:

- `time`: the forecast reference time, one value per cycle. New cycles are appended along it.
- `step`: the lead time from the cycle time, 0 to 72 hours in 25 steps, the same in every cycle.

Valid time is `time + step`, so it is never stored. No data is copied: the store holds only
pointers to byte ranges in NOAA's files.

In [ ]:
%pip install -q "icechunk>=2.2" "virtualizarr>=2.8" "xarray>=2026.7" "zarr>=3.4" obstore obspec_utils h5py h5netcdf s3fs "dask[array]"

## 1. Find three complete forecast cycles

The NOAA bucket is public, so we list it anonymously. To keep this quick we use the 00Z run of three
consecutive days, starting a week ago. A cycle is complete when all 25 files are there: `n006`, which is valid at
the cycle time (lead 0), and the forecast files `f003` to `f072`. In the file name, `f0NN` means valid
NN hours after the cycle time. The earlier nowcast file, `n003`, is left out. Keeping lead 0 as the
first step is what lets Rolodex build a best estimate from the store later.

In [ ]:
import datetime as dt
import re
import warnings

import s3fs

warnings.filterwarnings("ignore", category=UserWarning)

fs = s3fs.S3FileSystem(anon=True)
SRC = "noaa-nos-ofs-pds/gomofs/netcdf"
NAME = re.compile(r"gomofs\.t(\d\d)z\.(\d{8})\.fields\.([nf])(\d{3})\.nc$")
N_CYCLES = 3


start = dt.date.today() - dt.timedelta(days=7)
found = {}
for day in [start + dt.timedelta(days=i) for i in range(N_CYCLES + 2)]:   # spare days, in case one is incomplete
    for key in fs.glob(f"{SRC}/{day:%Y/%m/%d}/gomofs.t00z.{day:%Y%m%d}.fields.*.nc"):
        hour, ymd, kind, n = NAME.search(key).groups()
        if kind == "n" and n != "006":   # n003 is 3 h before the cycle time; keep only the one valid at it
            continue
        lead = 0 if kind == "n" else int(n)
        cycle = dt.datetime.strptime(ymd + hour, "%Y%m%d%H")
        found.setdefault(cycle, []).append((lead, key))

complete = {c: sorted(files) for c, files in sorted(found.items()) if len(files) == 25}
cycles = dict(list(complete.items())[:N_CYCLES])
for c, files in cycles.items():
    print(f"{c:%Y-%m-%d %Hz}: {len(files)} files, lead {files[0][0]} to {files[-1][0]} h")

## 2. Tell VirtualiZarr how to reach them

An object store for the bucket, a registry that maps the `s3://` prefix to it, and a parser for
the file format. These files are NetCDF4, which is HDF5 inside.

In [ ]:
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr.parsers import HDFParser

bucket = "s3://noaa-nos-ofs-pds/"
registry = ObjectStoreRegistry({bucket: S3Store(bucket="noaa-nos-ofs-pds", region="us-east-1", skip_signature=True)})
parser = HDFParser()

## 3. Open one file virtually

Only the metadata is read. The valid time, `ocean_time`, is loaded into memory; the big arrays
stay as `ManifestArray`s. Each file has a single time step. Grid and mask variables, like `h`
and `mask_rho`, have no time dimension at all.

In [ ]:
from virtualizarr import open_virtual_dataset

TIME_DIM = "ocean_time"


def open_file(key):
    ds = open_virtual_dataset(url=f"s3://{key}", parser=parser, registry=registry,
                              loadable_variables=[TIME_DIM], decode_times=True)
    return ds.drop_vars("ntimes", errors="ignore")   # a per-file scalar, misleading once merged


first_cycle = next(iter(cycles))
one = open_file(cycles[first_cycle][0][1])
one

## 4. Turn one cycle into `(time, step)`

For one cycle we open its 25 files and join them along the valid time. Then we reshape:

- The valid-time dimension becomes `step`, the lead time from the cycle time.
- `time` becomes a single forecast reference time. We add it **only** to the variables that vary
  with `step`. Adding it to the whole dataset would give every grid and mask variable a `time`
  dimension, stored once per cycle.
- We pin how `time` and `step` are encoded. Left to xarray, the units depend on each cycle's own
  values, and a cycle appended later can decode to the wrong date.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import xarray as xr


def open_cycle(cycle, files):
    """One cycle as (time=1, step, ...)."""
    with ThreadPoolExecutor(16) as pool:
        parts = list(pool.map(open_file, [key for _, key in files]))
    parts.sort(key=lambda d: d[TIME_DIM].values[0])
    ds = xr.concat(parts, dim=TIME_DIM, data_vars="minimal", coords="minimal",
                   compat="override", combine_attrs="override")

    # the file names and the file contents must agree, or the step axis would lie
    valid = pd.DatetimeIndex(ds[TIME_DIM].values)
    ref = pd.Timestamp(cycle)
    assert (valid == [ref + pd.Timedelta(hours=lead) for lead, _ in files]).all()

    ds = ds.rename_dims({TIME_DIM: "step"}).drop_vars(TIME_DIM)
    ds = ds.assign_coords(step=("step", (valid - ref).values, {"standard_name": "forecast_period"}))

    varying = [v for v in ds.data_vars if "step" in ds[v].dims]
    static = ds.drop_vars(varying)
    ds = xr.merge([ds[varying].expand_dims(time=[np.datetime64(ref)]), static],
                  compat="override", combine_attrs="override")
    ds["time"].attrs.update(standard_name="forecast_reference_time")
    ds["time"].encoding = {"units": "hours since 2024-01-01", "dtype": "int64"}
    ds["step"].encoding = {"units": "hours", "dtype": "int64"}
    return ds


one_cycle = open_cycle(first_cycle, cycles[first_cycle])
print(dict(one_cycle.sizes))
print("temp:", one_cycle["temp"].dims, "   h:", one_cycle["h"].dims)

## 5. Write the store: create with the first cycle, append the second

The store lives in a local folder. The *virtual chunk container* tells Icechunk that the chunks
live outside the store, in NOAA's bucket, and how to reach them.

A later cycle is **appended along `time`**. An append carries only the variables that have a
`time` dimension: writing the static variables again would rewrite their metadata, which
VirtualiZarr refuses. We commit once per cycle.

In [ ]:
import shutil

import icechunk

path = "gomofs-fmrc-icechunk"
shutil.rmtree(path, ignore_errors=True)   # start fresh each run

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=bucket,
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    )
)
repo = icechunk.Repository.create(icechunk.local_filesystem_storage(path), config)
repo.save_config()


def for_append(ds, dim):
    return ds.drop_vars([v for v in ds.variables if dim not in ds[v].dims])


def commit_cycle(repo, cycle, ds, append):
    session = repo.writable_session("main")
    if append:
        for_append(ds, "time").vz.to_icechunk(session.store, append_dim="time")
    else:
        ds.vz.to_icechunk(session.store)
    session.commit(f"GoMOFS cycle {cycle:%Y-%m-%d %Hz}")
    print(f"committed {cycle:%Y-%m-%d %Hz}")


first, second, third = cycles
commit_cycle(repo, first, one_cycle, append=False)                           # create
commit_cycle(repo, second, open_cycle(second, cycles[second]), append=True)   # append

## 6. Append another day

This is what a daily update does. Open the existing store, open the newest cycle the same way, and
append it along `time`. Everything already in the store stays as it is.

In [ ]:
creds = icechunk.containers_credentials({bucket: icechunk.s3_credentials(anonymous=True)})
repo = icechunk.Repository.open(icechunk.local_filesystem_storage(path), authorize_virtual_chunk_access=creds)

commit_cycle(repo, third, open_cycle(third, cycles[third]), append=True)

## 7. Open it as a reader would

A reader opens the store and says, explicitly, that it may fetch chunks from NOAA's bucket.
Then it is just an xarray dataset, with one `time` entry for each cycle.

In [ ]:
reader = icechunk.Repository.open(icechunk.local_filesystem_storage(path), authorize_virtual_chunk_access=creds)
ds = xr.open_zarr(reader.readonly_session("main").store, consolidated=False, chunks={})
ds

## 8. Check it

Four things should hold: one `time` per cycle, the lead times run from 0 to 72 hours, the grid
variables have no `time` dimension, and a value read through the store equals the same value read
from NOAA's file.

In [ ]:
print("cycles:", ds.time.values.astype("datetime64[h]"))
print("lead times (h):", (ds.step.values / np.timedelta64(1, "h")).astype(int))
print("variables that have a time dimension:", sorted(v for v in ds.data_vars if "time" in ds[v].dims)[:8], "...")
print("grid variables without one: h", ds["h"].dims, " mask_rho", ds["mask_rho"].dims)

ci, si = 2, 5
cycle, files = list(cycles.items())[ci]
key = files[si][1]
with fs.open(key) as f:
    raw = xr.open_dataset(f, engine="h5netcdf")
    want = raw["zeta"].isel({TIME_DIM: 0, "eta_rho": 100, "xi_rho": 150}).values
got = ds["zeta"].isel(time=ci, step=si, eta_rho=100, xi_rho=150).values
print(f"zeta, cycle {cycle:%Y-%m-%d %Hz} step {ds.step.values[si] / np.timedelta64(1, 'h'):.0f} h:",
      f"store {float(got):.4f}   NOAA file {float(want):.4f}")
assert np.allclose(got, want, equal_nan=True)

## How big is the store?

In [ ]:
import os

size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs_ in os.walk(path) for f in fs_)
n_files = sum(len(files) for files in cycles.values())
print(f"Icechunk store: {size / 1e3:.0f} kB, pointing at {n_files} NetCDF files "
      f"({sum(fs.size(key) for files in cycles.values() for _, key in files) / 1e9:.1f} GB)")

## Next

The store keeps every run. To pull out one forecast, or a continuous *best estimate* that stitches
the newest hours from each run, see the next notebook, which reads it with
[Rolodex](https://github.com/dcherian/rolodex).